In [ ]:
import os
import shutil
import pandas as pd
from tqdm import tqdm
base_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/u2net_next_inpainting/HPDv3"
base_target_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/pick-a-pic-v2/DiffusionDRO-HPDv3-top_500_pickscore_images"
os.makedirs(base_target_dir, exist_ok=True)

all_information_df = pd.read_csv("/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/HPDv3/real_images_uid_prompt.csv", dtype={"uid": str})
pickscore_df = pd.read_csv(os.path.join(base_dir, "pickscore/pickscore_score.csv"), dtype={"uid": str})
anime_df = pd.read_csv("/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/u2net_next_inpainting/HPDv3/qwen3_anime/qwen3_anime.csv", dtype={"uid": str})
valid_real_image_uids = anime_df[anime_df['anime'] == 'no']['uid'].tolist()
pickscore_df = pickscore_df[pickscore_df['uid'].isin(valid_real_image_uids)]

pickscore_df = pickscore_df.sort_values(by='real_image_score', ascending=False)
pickscore_df = pickscore_df.head(500)

for idx, row in tqdm(pickscore_df.iterrows()):
    uid = row['uid']
    prompt = all_information_df[all_information_df['uid'] == uid]['prompt'].values[0]
    real_image_path = os.path.join(base_dir, "real", f"{uid}.png")
    target_dir = os.path.join(base_target_dir, uid)
    os.makedirs(target_dir, exist_ok=True)
    
    ## copy image ##
    target_image_path = os.path.join(target_dir, f"{uid}.png")
    shutil.copy(real_image_path, target_image_path)
    
    ## write caption ##
    caption_path = os.path.join(target_dir, "caption.txt")
    with open(caption_path, "w") as f:
        f.write(prompt)
    